# AlexNet — Implementation from Scratch
# AlexNet — 처음부터 구현하기

**Paper**: Krizhevsky, Sutskever & Hinton, "ImageNet Classification with Deep Convolutional Neural Networks" (2012)

이 노트북에서는 AlexNet의 핵심 구성 요소를 NumPy로 직접 구현한 뒤, PyTorch로 검증합니다.

This notebook implements AlexNet's key components from scratch in NumPy, then validates with PyTorch.

**목차 / Contents:**
1. ReLU vs Sigmoid/Tanh — 활성화 함수 비교 / Activation function comparison
2. Convolution from Scratch — 합성곱 직접 구현 / Manual convolution implementation
3. Max Pooling (Overlapping) — 겹치는 최대 풀링
4. Local Response Normalization (LRN) — 지역 응답 정규화
5. Dropout from Scratch — 드롭아웃 직접 구현
6. PCA Color Augmentation — PCA 색상 증강
7. AlexNet Architecture (PyTorch) — 전체 아키텍처 구현
8. Feature Visualization & Learned Representations — 특징 시각화

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

---

## Part 1: ReLU vs Sigmoid vs Tanh — 활성화 함수 비교
## Part 1: ReLU vs Sigmoid vs Tanh — Activation Function Comparison

AlexNet의 가장 중요한 기술적 기여는 ReLU의 채택입니다.
Sigmoid/Tanh는 **saturating nonlinearity**로, 큰 입력에서 gradient가 0에 수렴합니다.
ReLU $f(x) = \max(0, x)$는 양수 영역에서 gradient가 항상 1입니다.

AlexNet's most important technical contribution is the adoption of ReLU.
Sigmoid/Tanh are **saturating nonlinearities** with gradients approaching 0 for large inputs.
ReLU $f(x) = \max(0, x)$ has gradient always equal to 1 in the positive region.

In [ ]:
def sigmoid(x):
    """Sigmoid activation function."""
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))


def sigmoid_grad(x):
    """Gradient of sigmoid."""
    s = sigmoid(x)
    return s * (1 - s)


def tanh_grad(x):
    """Gradient of tanh."""
    return 1 - np.tanh(x) ** 2


def relu(x):
    """ReLU activation function: f(x) = max(0, x)."""
    return np.maximum(0, x)


def relu_grad(x):
    """Gradient of ReLU."""
    return (x > 0).astype(float)


# --- Visualization ---
x = np.linspace(-6, 6, 500)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Activations
axes[0].plot(x, sigmoid(x), label='Sigmoid', linewidth=2)
axes[0].plot(x, np.tanh(x), label='Tanh', linewidth=2)
axes[0].plot(x, relu(x), label='ReLU', linewidth=2, color='red')
axes[0].set_title('Activation Functions / 활성화 함수', fontsize=13)
axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')
axes[0].legend(fontsize=11)
axes[0].set_ylim(-1.5, 4)
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.3)
axes[0].axvline(0, color='gray', linestyle='--', alpha=0.3)
axes[0].grid(True, alpha=0.3)

# Gradients
axes[1].plot(x, sigmoid_grad(x), label="Sigmoid' (max=0.25)", linewidth=2)
axes[1].plot(x, tanh_grad(x), label="Tanh' (max=1.0)", linewidth=2)
axes[1].plot(x, relu_grad(x), label="ReLU' (1 if x>0)", linewidth=2, color='red')
axes[1].set_title('Gradients / 기울기', fontsize=13)
axes[1].set_xlabel('x')
axes[1].set_ylabel("f'(x)")
axes[1].legend(fontsize=11)
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.3)
axes[1].axvline(0, color='gray', linestyle='--', alpha=0.3)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle('Why ReLU? — Saturating vs Non-saturating Nonlinearities',
             fontsize=14, fontweight='bold', y=1.02)
plt.show()

# Gradient vanishing demonstration through layers
n_layers = 10
x_val = 0.5
sigmoid_chain = [sigmoid_grad(x_val) ** n for n in range(1, n_layers + 1)]
tanh_chain = [tanh_grad(x_val) ** n for n in range(1, n_layers + 1)]
relu_chain = [relu_grad(x_val) ** n for n in range(1, n_layers + 1)]

fig, ax = plt.subplots(figsize=(10, 5))
layers = range(1, n_layers + 1)
ax.semilogy(layers, sigmoid_chain, 'o-', label=f'Sigmoid (x={x_val})', linewidth=2)
ax.semilogy(layers, tanh_chain, 's-', label=f'Tanh (x={x_val})', linewidth=2)
ax.semilogy(layers, relu_chain, '^-', label=f'ReLU (x={x_val})', linewidth=2, color='red')
ax.set_xlabel('Number of layers / 층 수')
ax.set_ylabel('Gradient magnitude / 기울기 크기 (log scale)')
ax.set_title('Vanishing Gradient Problem / 기울기 소실 문제', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"After {n_layers} layers:")
print(f"  Sigmoid gradient: {sigmoid_chain[-1]:.2e}")
print(f"  Tanh gradient:    {tanh_chain[-1]:.2e}")
print(f"  ReLU gradient:    {relu_chain[-1]:.2e}")

---

## Part 2: Convolution from Scratch — 합성곱 직접 구현
## Part 2: Convolution from Scratch

AlexNet의 Conv1은 224×224×3 입력에 96개의 11×11×3 필터를 stride 4로 적용합니다.
출력 크기: $\lfloor(224-11)/4\rfloor + 1 = 55$, 즉 55×55×96.

여기서는 단순한 2D convolution을 직접 구현하여 원리를 이해합니다.

AlexNet's Conv1 applies 96 filters of 11×11×3 with stride 4 to 224×224×3 input.
Output size: $\lfloor(224-11)/4\rfloor + 1 = 55$, i.e., 55×55×96.

We implement simple 2D convolution from scratch to understand the mechanics.

In [ ]:
def conv2d(input_img, kernels, bias, stride=1, padding=0):
    """2D convolution (multi-channel input, multiple filters).

    Args:
        input_img: Input image, shape (C_in, H, W).
        kernels: Filter weights, shape (C_out, C_in, kH, kW).
        bias: Bias for each filter, shape (C_out,).
        stride: Convolution stride.
        padding: Zero-padding added to each side.

    Returns:
        Output feature maps, shape (C_out, H_out, W_out).
    """
    c_in, h_in, w_in = input_img.shape
    c_out, _, kh, kw = kernels.shape

    # Apply padding
    if padding > 0:
        input_img = np.pad(input_img,
                           ((0, 0), (padding, padding), (padding, padding)),
                           mode='constant')
        _, h_in, w_in = input_img.shape

    h_out = (h_in - kh) // stride + 1
    w_out = (w_in - kw) // stride + 1
    output = np.zeros((c_out, h_out, w_out))

    for f in range(c_out):
        for i in range(h_out):
            for j in range(w_out):
                h_start = i * stride
                w_start = j * stride
                receptive_field = input_img[:, h_start:h_start+kh,
                                           w_start:w_start+kw]
                output[f, i, j] = np.sum(
                    receptive_field * kernels[f]) + bias[f]

    return output


# --- Demonstrate Conv1 dimensions ---
# Simulate AlexNet Conv1: 3×224×224 -> 96×55×55
print("=== AlexNet Conv1 Dimension Check ===")
print(f"Input:  3 × 224 × 224 = {3*224*224:,} values")

h_out = (224 - 11) // 4 + 1
print(f"Output: 96 × {h_out} × {h_out} = {96*h_out*h_out:,} values")
print(f"Params: 96 × 3 × 11 × 11 + 96(bias) = {96*3*11*11 + 96:,}")
print()

# Small demo: 3×16×16 input, 4 filters of 3×5×5, stride 2
demo_input = np.random.randn(3, 16, 16)
demo_kernels = np.random.randn(4, 3, 5, 5) * 0.01
demo_bias = np.zeros(4)

output = conv2d(demo_input, demo_kernels, demo_bias, stride=2)
expected_h = (16 - 5) // 2 + 1
print(f"Demo: input (3,16,16) + 4 filters (3,5,5) stride 2")
print(f"Output shape: {output.shape} (expected: (4, {expected_h}, {expected_h}))")

# Visualize a simple edge detection
# Create a simple grayscale image with vertical and horizontal edges
simple_img = np.zeros((1, 32, 32))
simple_img[0, 5:27, 10:14] = 1.0  # Vertical bar
simple_img[0, 14:18, 5:27] = 1.0  # Horizontal bar (cross)

# Vertical edge detector
vert_kernel = np.zeros((1, 1, 3, 3))
vert_kernel[0, 0] = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])

# Horizontal edge detector
horiz_kernel = np.zeros((1, 1, 3, 3))
horiz_kernel[0, 0] = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]])

vert_edges = conv2d(simple_img, vert_kernel, np.zeros(1), stride=1, padding=1)
horiz_edges = conv2d(simple_img, horiz_kernel, np.zeros(1), stride=1, padding=1)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(simple_img[0], cmap='gray')
axes[0].set_title('Input Image / 입력 이미지')
axes[1].imshow(vert_edges[0], cmap='RdBu_r')
axes[1].set_title('Vertical Edge Filter / 수직 경계 필터')
axes[2].imshow(horiz_edges[0], cmap='RdBu_r')
axes[2].set_title('Horizontal Edge Filter / 수평 경계 필터')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

---

## Part 3: Max Pooling (Overlapping) — 겹치는 최대 풀링
## Part 3: Max Pooling (Overlapping)

AlexNet은 **overlapping pooling**을 사용합니다: pool size $z=3$, stride $s=2$.
전통적 pooling ($s=z$)과 달리 풀링 영역이 겹칩니다.
Top-1/Top-5 error를 0.4%/0.3% 감소시킵니다.

AlexNet uses **overlapping pooling**: pool size $z=3$, stride $s=2$.
Unlike traditional pooling ($s=z$), pooling regions overlap.
Reduces top-1/top-5 error by 0.4%/0.3%.

In [ ]:
def max_pool2d(input_feat, pool_size=3, stride=2):
    """Max pooling (supports overlapping when pool_size > stride).

    Args:
        input_feat: Input feature maps, shape (C, H, W).
        pool_size: Size of pooling window.
        stride: Stride of pooling.

    Returns:
        Pooled output, shape (C, H_out, W_out).
    """
    c, h_in, w_in = input_feat.shape
    h_out = (h_in - pool_size) // stride + 1
    w_out = (w_in - pool_size) // stride + 1
    output = np.zeros((c, h_out, w_out))

    for i in range(h_out):
        for j in range(w_out):
            h_start = i * stride
            w_start = j * stride
            window = input_feat[:, h_start:h_start+pool_size,
                                w_start:w_start+pool_size]
            output[:, i, j] = window.reshape(c, -1).max(axis=1)

    return output


# --- Demonstrate AlexNet pooling dimensions ---
print("=== AlexNet Pooling Dimensions ===")
# After Conv1+ReLU: 96×55×55
# After LRN + MaxPool(3,stride=2): 96×27×27
pool_in = (55 - 3) // 2 + 1
print(f"After Conv1: 96×55×55")
print(f"After Pool1 (3×3, stride 2): 96×{pool_in}×{pool_in}")
print()

# Compare overlapping vs non-overlapping
feat = np.random.randn(4, 13, 13)  # Simulate feature maps

pool_overlap = max_pool2d(feat, pool_size=3, stride=2)   # Overlapping
pool_nonoverlap = max_pool2d(feat, pool_size=2, stride=2)  # Non-overlapping

print(f"Input: {feat.shape}")
print(f"Overlapping pool (z=3, s=2): {pool_overlap.shape}")
print(f"Non-overlapping pool (z=2, s=2): {pool_nonoverlap.shape}")

# Visualize the overlap
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax_idx, (title, z, s) in enumerate([
    ('Non-overlapping (z=2, s=2)', 2, 2),
    ('Overlapping (z=3, s=2) — AlexNet', 3, 2),
]):
    ax = axes[ax_idx]
    grid = np.zeros((8, 8))
    colors = plt.cm.Set3(np.linspace(0, 1, 16))
    ax.imshow(grid, cmap='gray_r', alpha=0.1)

    pool_idx = 0
    for pi in range(0, 8 - z + 1, s):
        for pj in range(0, 8 - z + 1, s):
            color = colors[pool_idx % len(colors)]
            rect = plt.Rectangle((pj - 0.5, pi - 0.5), z, z,
                                 fill=True, facecolor=color, alpha=0.3,
                                 edgecolor='black', linewidth=2)
            ax.add_patch(rect)
            ax.text(pj + z/2 - 0.5, pi + z/2 - 0.5, str(pool_idx),
                    ha='center', va='center', fontsize=10, fontweight='bold')
            pool_idx += 1

    ax.set_xlim(-0.6, 7.6)
    ax.set_ylim(7.6, -0.6)
    ax.set_title(title, fontsize=12)
    ax.set_xticks(range(8))
    ax.set_yticks(range(8))
    ax.grid(True, alpha=0.3)

plt.suptitle('Pooling Regions: Overlapping vs Non-overlapping\n'
             '풀링 영역: 겹침 vs 비겹침', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Part 4: Local Response Normalization (LRN) — 지역 응답 정규화
## Part 4: Local Response Normalization (LRN)

$$b^i_{x,y} = \frac{a^i_{x,y}}{\left(k + \alpha \sum_{j=\max(0,i-n/2)}^{\min(N-1,i+n/2)} (a^j_{x,y})^2\right)^\beta}$$

생물학적 lateral inhibition에서 영감: 인접 커널 간 경쟁을 유발합니다.
현대 네트워크에서는 Batch Normalization으로 대체되었지만, 2012년에는 top-1 error를 1.4% 감소시켰습니다.

Inspired by biological lateral inhibition: creates competition between adjacent kernels.
Replaced by Batch Normalization in modern networks, but reduced top-1 error by 1.4% in 2012.

In [ ]:
def local_response_norm(input_feat, n=5, k=2, alpha=1e-4, beta=0.75):
    """Local Response Normalization (across channels).

    Args:
        input_feat: Feature maps after ReLU, shape (C, H, W).
        n: Number of adjacent channels to normalize over.
        k: Bias constant.
        alpha: Scaling constant.
        beta: Exponent.

    Returns:
        Normalized feature maps, same shape as input.
    """
    c, h, w = input_feat.shape
    output = np.zeros_like(input_feat)
    half_n = n // 2

    for i in range(c):
        j_start = max(0, i - half_n)
        j_end = min(c, i + half_n + 1)
        sq_sum = np.sum(input_feat[j_start:j_end] ** 2, axis=0)
        output[i] = input_feat[i] / (k + alpha * sq_sum) ** beta

    return output


# --- Demonstrate LRN effect ---
# Create synthetic feature maps: one channel has strong activation,
# adjacent channels should be suppressed
n_channels = 8
feat_size = 10
feat = np.zeros((n_channels, feat_size, feat_size))

# Channel 3 has strong activation in center
feat[3, 3:7, 3:7] = 5.0
# Channel 4 has moderate activation in same area
feat[4, 3:7, 3:7] = 2.0
# Channel 5 has weak activation
feat[5, 3:7, 3:7] = 1.0

feat_normed = local_response_norm(feat, n=5, k=2, alpha=1e-4, beta=0.75)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for idx, ch in enumerate([2, 3, 4, 5]):
    axes[0, idx].imshow(feat[ch], cmap='hot', vmin=0, vmax=5)
    axes[0, idx].set_title(f'Before LRN: Ch {ch}')
    axes[0, idx].axis('off')
    
    axes[1, idx].imshow(feat_normed[ch], cmap='hot', vmin=0, vmax=5)
    axes[1, idx].set_title(f'After LRN: Ch {ch}')
    axes[1, idx].axis('off')

# Print suppression ratios
print("=== LRN Suppression Effect (center pixel) ===")
for ch in [2, 3, 4, 5]:
    before = feat[ch, 5, 5]
    after = feat_normed[ch, 5, 5]
    ratio = after / before if before > 0 else 0
    print(f"  Channel {ch}: {before:.2f} -> {after:.4f} "
          f"(suppression: {1-ratio:.1%})")

plt.suptitle('Local Response Normalization (Lateral Inhibition)\n'
             '지역 응답 정규화 (측면 억제)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Part 5: Dropout from Scratch — 드롭아웃 직접 구현
## Part 5: Dropout from Scratch

학습 시 각 뉴런을 확률 $p=0.5$로 무작위 비활성화합니다.
테스트 시 모든 뉴런을 사용하되 출력에 $(1-p)$를 곱합니다.
효과: 지수적으로 많은 서브네트워크의 앙상블을 근사합니다.

During training, randomly deactivate each neuron with probability $p=0.5$.
At test time, use all neurons but multiply outputs by $(1-p)$.
Effect: approximates an ensemble of exponentially many subnetworks.

In [ ]:
def dropout_train(x, p=0.5, rng=None):
    """Apply dropout during training (inverted dropout).

    Args:
        x: Input activations.
        p: Probability of keeping each neuron (not dropping).
        rng: Random number generator.

    Returns:
        Tuple of (dropped_output, mask).
    """
    if rng is None:
        rng = np.random.default_rng()
    mask = (rng.random(x.shape) < p).astype(float)
    return x * mask / p, mask  # Inverted dropout: scale by 1/p during training


def dropout_test(x):
    """At test time with inverted dropout, no scaling needed."""
    return x


# --- AlexNet-style dropout (original, not inverted) ---
def alexnet_dropout_train(x, p_drop=0.5, rng=None):
    """Original AlexNet dropout (scale at test time, not train time)."""
    if rng is None:
        rng = np.random.default_rng()
    mask = (rng.random(x.shape) >= p_drop).astype(float)
    return x * mask, mask


def alexnet_dropout_test(x, p_drop=0.5):
    """Original AlexNet: scale by (1-p_drop) at test time."""
    return x * (1 - p_drop)


# --- Visualize dropout effect ---
rng = np.random.default_rng(42)
fc_activations = rng.standard_normal(4096) * 2  # Simulate FC layer
fc_activations = np.maximum(0, fc_activations)    # After ReLU

# Apply dropout 5 times to show different "thinned" networks
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Original activations
axes[0, 0].bar(range(100), fc_activations[:100], color='blue', alpha=0.7)
axes[0, 0].set_title('Original (no dropout)', fontsize=11)
axes[0, 0].set_ylabel('Activation')

for trial in range(5):
    ax = axes[(trial + 1) // 3, (trial + 1) % 3]
    dropped, mask = alexnet_dropout_train(fc_activations, p_drop=0.5, rng=rng)
    colors = ['red' if m == 0 else 'blue' for m in mask[:100]]
    ax.bar(range(100), fc_activations[:100], color=colors, alpha=0.7)
    active_pct = mask.mean() * 100
    ax.set_title(f'Dropout trial {trial+1} ({active_pct:.0f}% active)',
                 fontsize=11)

plt.suptitle('Dropout: Different "thinned" networks each forward pass\n'
             '드롭아웃: 매번 다른 "축소된" 네트워크 (first 100 neurons shown)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Count possible subnetworks
print(f"FC6+FC7 neurons: 4096 + 4096 = 8,192")
print(f"Possible subnetworks: 2^8192 ≈ 10^{8192 * np.log10(2):.0f}")
print(f"  (more than atoms in the observable universe: ~10^80)")

---

## Part 6: PCA Color Augmentation — PCA 색상 증강
## Part 6: PCA Color Augmentation

각 학습 이미지의 모든 픽셀에 다음을 추가합니다:

$$\Delta \mathbf{c} = [\mathbf{p}_1, \mathbf{p}_2, \mathbf{p}_3][\alpha_1 \lambda_1, \alpha_2 \lambda_2, \alpha_3 \lambda_3]^T, \quad \alpha_i \sim \mathcal{N}(0, 0.1)$$

자연 이미지의 조명 변화가 RGB 주성분 방향을 따른다는 관찰에 기반합니다.
Top-1 error를 1% 이상 감소시킵니다.

Add to every pixel of each training image. Based on the observation that illumination
changes follow RGB principal component directions. Reduces top-1 error by >1%.

In [ ]:
def compute_rgb_pca(images):
    """Compute PCA of RGB pixel values across a set of images.

    Args:
        images: Array of shape (N, H, W, 3), pixel values in [0, 255].

    Returns:
        eigenvectors: Shape (3, 3), columns are principal components.
        eigenvalues: Shape (3,), sorted descending.
    """
    # Reshape to (N*H*W, 3)
    pixels = images.reshape(-1, 3).astype(np.float64)

    # Compute covariance matrix
    mean = pixels.mean(axis=0)
    centered = pixels - mean
    cov = np.cov(centered.T)  # 3×3 covariance matrix

    # Eigendecomposition
    eigenvalues, eigenvectors = np.linalg.eigh(cov)

    # Sort descending
    idx = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]

    return eigenvectors, eigenvalues


def pca_color_augmentation(image, eigenvectors, eigenvalues, alpha_std=0.1):
    """Apply PCA color augmentation to a single image.

    Args:
        image: Shape (H, W, 3), pixel values in [0, 255].
        eigenvectors: Shape (3, 3) from compute_rgb_pca.
        eigenvalues: Shape (3,) from compute_rgb_pca.
        alpha_std: Standard deviation of random alpha.

    Returns:
        Augmented image, same shape.
    """
    alphas = np.random.normal(0, alpha_std, size=3)
    delta = eigenvectors @ (alphas * eigenvalues)
    augmented = image + delta[np.newaxis, np.newaxis, :]
    return np.clip(augmented, 0, 255).astype(np.uint8)


# --- Create a synthetic natural image for demonstration ---
h, w = 128, 128
rng = np.random.default_rng(42)

# Simulate a simple natural scene: blue sky + green grass
img = np.zeros((h, w, 3), dtype=np.float64)
# Sky (top half): bluish
img[:h//2, :, 0] = 135 + rng.normal(0, 10, (h//2, w))  # R
img[:h//2, :, 1] = 206 + rng.normal(0, 10, (h//2, w))  # G
img[:h//2, :, 2] = 235 + rng.normal(0, 10, (h//2, w))  # B
# Grass (bottom half): greenish
img[h//2:, :, 0] = 34 + rng.normal(0, 15, (h//2, w))
img[h//2:, :, 1] = 139 + rng.normal(0, 15, (h//2, w))
img[h//2:, :, 2] = 34 + rng.normal(0, 15, (h//2, w))
img = np.clip(img, 0, 255)

# Compute PCA
evecs, evals = compute_rgb_pca(img[np.newaxis])

print("=== RGB PCA Results ===")
print(f"Eigenvalues: {evals}")
print(f"Explained variance ratio: {evals / evals.sum()}")
print(f"\nPrincipal components (columns):")
for i in range(3):
    print(f"  PC{i+1}: [{evecs[0,i]:.3f}, {evecs[1,i]:.3f}, {evecs[2,i]:.3f}]")

# Show original and augmented versions
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes[0, 0].imshow(img.astype(np.uint8))
axes[0, 0].set_title('Original / 원본', fontsize=11)

np.random.seed(0)
for i in range(7):
    row, col = (i + 1) // 4, (i + 1) % 4
    augmented = pca_color_augmentation(img, evecs, evals, alpha_std=0.1)
    axes[row, col].imshow(augmented)
    axes[row, col].set_title(f'Augmented {i+1}', fontsize=11)

for ax in axes.flat:
    ax.axis('off')

plt.suptitle('PCA Color Augmentation — Simulating Illumination Changes\n'
             'PCA 색상 증강 — 조명 변화 시뮬레이션 (α~N(0,0.1))',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Part 7: AlexNet Architecture (PyTorch) — 전체 아키텍처 구현
## Part 7: AlexNet Architecture (PyTorch)

전체 AlexNet을 PyTorch로 구현합니다.
GPU 분할은 현대적으로는 불필요하므로, 단일 네트워크로 구현합니다.
CIFAR-10 (32×32)에 맞게 수정된 "Mini-AlexNet"도 함께 구현합니다.

Full AlexNet implementation in PyTorch.
GPU split is unnecessary on modern hardware, so we implement as a single network.
Also implement a "Mini-AlexNet" adapted for CIFAR-10 (32×32).

In [ ]:
import torch
import torch.nn as nn


class AlexNet(nn.Module):
    """Original AlexNet architecture (single-GPU version).

    Input: 224×224×3 RGB images.
    Output: 1000-class probability distribution.
    Parameters: ~60 million.
    """

    def __init__(self, num_classes=1000):
        super().__init__()
        self.features = nn.Sequential(
            # Conv1: 3×224×224 -> 96×55×55
            nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2),
            nn.MaxPool2d(kernel_size=3, stride=2),  # -> 96×27×27

            # Conv2: 96×27×27 -> 256×27×27
            nn.Conv2d(96, 256, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2),
            nn.MaxPool2d(kernel_size=3, stride=2),  # -> 256×13×13

            # Conv3: 256×13×13 -> 384×13×13
            nn.Conv2d(256, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            # Conv4: 384×13×13 -> 384×13×13
            nn.Conv2d(384, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            # Conv5: 384×13×13 -> 256×13×13
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),  # -> 256×6×6
        )

        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


# --- Verify architecture ---
model = AlexNet(num_classes=1000)
dummy_input = torch.randn(1, 3, 224, 224)
output = model(dummy_input)

print("=== AlexNet Architecture ===")
print(f"Input:  {dummy_input.shape}")
print(f"Output: {output.shape}")
print()

# Count parameters per layer
total_params = 0
print(f"{'Layer':<30} {'Shape':<25} {'Parameters':>12}")
print("-" * 70)
for name, param in model.named_parameters():
    n_params = param.numel()
    total_params += n_params
    print(f"{name:<30} {str(list(param.shape)):<25} {n_params:>12,}")
print("-" * 70)
print(f"{'Total':<55} {total_params:>12,}")

---

## Part 8: Feature Visualization & Training Demo — 특징 시각화 & 학습 데모
## Part 8: Feature Visualization & Training Demo

AlexNet의 핵심 결과 중 하나는 Conv1이 학습한 필터들이 Gabor-like edge와 color blob을
자동으로 학습한다는 것입니다. 여기서는 Mini-AlexNet을 CIFAR-10에서 간단히 학습하고,
학습된 필터를 시각화하며, 각 기법(ReLU, Dropout 등)의 효과를 비교합니다.

A key AlexNet result is that Conv1 filters automatically learn Gabor-like edges and color
blobs. Here we briefly train a Mini-AlexNet on CIFAR-10, visualize learned filters,
and compare the effect of each technique (ReLU, Dropout, etc.).

In [ ]:
import torch.optim as optim
from torchvision import datasets, transforms


class MiniAlexNet(nn.Module):
    """Mini-AlexNet adapted for CIFAR-10 (32×32×3 input)."""

    def __init__(self, num_classes=10, use_relu=True, use_dropout=True,
                 use_lrn=True):
        super().__init__()
        act_fn = nn.ReLU(inplace=True) if use_relu else nn.Tanh()

        layers = [
            nn.Conv2d(3, 64, kernel_size=5, stride=1, padding=2),
            act_fn if use_relu else nn.Tanh(),
        ]
        if use_lrn:
            layers.append(
                nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2))
        layers.append(nn.MaxPool2d(kernel_size=3, stride=2))  # -> 64×15×15

        layers.extend([
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True) if use_relu else nn.Tanh(),
        ])
        if use_lrn:
            layers.append(
                nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2))
        layers.append(nn.MaxPool2d(kernel_size=3, stride=2))  # -> 128×7×7

        layers.extend([
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True) if use_relu else nn.Tanh(),
            nn.MaxPool2d(kernel_size=3, stride=2),  # -> 256×3×3
        ])

        self.features = nn.Sequential(*layers)

        clf_layers = []
        if use_dropout:
            clf_layers.append(nn.Dropout(0.5))
        clf_layers.extend([
            nn.Linear(256 * 3 * 3, 512),
            nn.ReLU(inplace=True) if use_relu else nn.Tanh(),
        ])
        if use_dropout:
            clf_layers.append(nn.Dropout(0.5))
        clf_layers.extend([
            nn.Linear(512, num_classes),
        ])
        self.classifier = nn.Sequential(*clf_layers)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


def train_epoch(model, loader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    total_loss, correct, total = 0, 0, 0
    for data, target in loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.size(0)
        correct += (output.argmax(1) == target).sum().item()
        total += data.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    """Evaluate on validation/test set."""
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            total_loss += loss.item() * data.size(0)
            correct += (output.argmax(1) == target).sum().item()
            total += data.size(0)
    return total_loss / total, correct / total


# --- Setup ---
device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available()
                      else 'cpu')
print(f"Device: {device}")

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

train_dataset = datasets.CIFAR10(root='./data', train=True,
                                 download=True, transform=transform_train)
test_dataset = datasets.CIFAR10(root='./data', train=False,
                                download=True, transform=transform_test)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=128, shuffle=False, num_workers=2)

print(f"Training samples: {len(train_dataset):,}")
print(f"Test samples:     {len(test_dataset):,}")

In [ ]:
# --- Compare ReLU vs Tanh, with/without Dropout ---
configs = {
    'ReLU + Dropout (AlexNet)': dict(use_relu=True, use_dropout=True,
                                     use_lrn=True),
    'ReLU, no Dropout':        dict(use_relu=True, use_dropout=False,
                                     use_lrn=True),
    'Tanh + Dropout':          dict(use_relu=False, use_dropout=True,
                                     use_lrn=False),
}

n_epochs = 15
results = {}

for name, cfg in configs.items():
    print(f"\n=== Training: {name} ===")
    model = MiniAlexNet(**cfg).to(device)
    optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9,
                          weight_decay=0.0005)
    criterion = nn.CrossEntropyLoss()

    train_accs, test_accs = [], []
    for epoch in range(n_epochs):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer,
                                            criterion, device)
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)
        train_accs.append(train_acc)
        test_accs.append(test_acc)

        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1:2d}: train_acc={train_acc:.3f} "
                  f"test_acc={test_acc:.3f}")

    results[name] = {'train': train_accs, 'test': test_accs, 'model': model}

# Plot results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, res in results.items():
    axes[0].plot(range(1, n_epochs+1), res['train'], '-o', label=name,
                 markersize=3)
    axes[1].plot(range(1, n_epochs+1), res['test'], '-o', label=name,
                 markersize=3)

axes[0].set_title('Training Accuracy / 학습 정확도')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Test Accuracy / 테스트 정확도')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Effect of ReLU and Dropout on CIFAR-10\n'
             'ReLU와 Dropout의 효과 (CIFAR-10)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Visualize learned Conv1 filters ---
best_model = results['ReLU + Dropout (AlexNet)']['model']

# Get Conv1 weights
conv1_weights = best_model.features[0].weight.data.cpu().numpy()
n_filters = conv1_weights.shape[0]

# Normalize each filter to [0, 1] for visualization
def normalize_filter(f):
    """Normalize filter to [0, 1] for display."""
    f = f.transpose(1, 2, 0)  # (C,H,W) -> (H,W,C)
    f = (f - f.min()) / (f.max() - f.min() + 1e-8)
    return f


fig, axes = plt.subplots(8, 8, figsize=(12, 12))
for i in range(min(64, n_filters)):
    ax = axes[i // 8, i % 8]
    filt = normalize_filter(conv1_weights[i])
    ax.imshow(filt)
    ax.axis('off')

plt.suptitle('Learned Conv1 Filters (Mini-AlexNet on CIFAR-10)\n'
             '학습된 Conv1 필터 (Mini-AlexNet, CIFAR-10)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Compare with AlexNet's Figure 3:")
print("  - Edge detectors (various orientations)")
print("  - Color blob detectors")
print("  - Frequency patterns")
print("Note: CIFAR-10 (32×32) filters are smaller (5×5) than")
print("AlexNet's (11×11), but similar patterns emerge.")

---

## 정리 / Summary

| Part | 구현 / Implementation | 핵심 발견 / Key Finding |
|---|---|---|
| 1 | ReLU vs Sigmoid/Tanh | ReLU는 gradient 소실 없이 deep network 학습 가능 / ReLU enables deep network training without gradient vanishing |
| 2 | Convolution from scratch | CNN은 이미지의 지역적 패턴을 계층적으로 학습 / CNNs hierarchically learn local image patterns |
| 3 | Overlapping MaxPool | 겹치는 풀링이 약간의 정규화 효과 제공 / Overlapping pooling provides slight regularization |
| 4 | LRN | 인접 채널 간 경쟁으로 일반화 향상 (현재는 BatchNorm으로 대체) / Competition between adjacent channels improves generalization (now replaced by BatchNorm) |
| 5 | Dropout | 지수적 앙상블 근사로 overfitting 방지 / Exponential ensemble approximation prevents overfitting |
| 6 | PCA Color Aug | 자연 이미지의 조명 불변성 학습 / Learning illumination invariance of natural images |
| 7 | Full AlexNet (PyTorch) | 60M 파라미터, 97.7%가 FC 층에 집중 / 60M parameters, 97.7% in FC layers |
| 8 | Training & Visualization | ReLU + Dropout 조합이 최고 성능 / ReLU + Dropout combination achieves best performance |

### 다음 논문과의 연결 / Connection to Next Papers

| 논문 / Paper | AlexNet과의 관계 / Relation to AlexNet |
|---|---|
| #14 Mikolov et al. (2013) — Word2Vec | AlexNet이 vision에서 보여준 "learned representation"을 NLP에 적용 / Applied AlexNet's "learned representation" success to NLP |
| #15 Kingma & Welling (2013) — VAE | AlexNet의 encoder 구조를 생성 모델에 활용 / Leveraged AlexNet's encoder structure for generative models |
| #19 He et al. (2015) — ResNet | AlexNet의 "깊이가 중요하다"를 극한까지 확장 (152층) / Extended AlexNet's "depth matters" to the extreme (152 layers) |